## Exercise 5: Geospatial wrangling and making maps

Skills: 
* More geospatial practice building on earlier skills
* Make a map with `folium` or `ipyleaflet` using `shared_utils.map_utils`

References: 
* https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html
* https://docs.calitp.org/data-infra/analytics_tools/python_libraries.html
* https://docs.calitp.org/data-infra/analytics_examples/warehouse_tutorial.html
* https://docs.calitp.org/data-infra/analytics_examples/new_tutorial.html

In [182]:
import geopandas as gpd
import intake
import pandas as pd

from calitp.tables import tbl
from calitp import query_sql
from siuba import *

from shapely.geometry import Polygon, LineString, Point
from shapely import wkt
from shapely.geometry import Point, Polygon
from shapely.wkt import loads

import shared_utils
import altair as alt
from shared_utils import altair_utils 

import branca
# Hint: if this doesn't import: refer to docs for correctly import
# cd into _shared_utils folder, run the make setup_env command

pd.options.display.float_format = "{:.2f}".format

## Research Question: What's the average number of trips per stop by operators in southern California? Show visualizations at the operator and county-level.
<br>**Geographic scope:** southern California counties
<br>**Deliverables:** chart(s) and map(s) showing metrics comparing across counties and also across operators. Make these visualizations using function(s).

### Prep data

* Use the same query, but grab a different set of operators. These are in southern California, so the map should zoom in counties ranging from LA to SD.
* To find what ITP IDs are what operators:
[agencies.yml](https://github.com/cal-itp/data-infra/blob/main/airflow/data/agencies.yml)
* *Hint*: for some counties, there are multiple operators. Make sure the average trips per stop by counties is the weighted average.
* Use the same [shapefile for CA counties](https://gis.data.ca.gov/datasets/CALFIRE-Forestry::california-county-boundaries/explore?location=37.246136%2C-119.002032%2C6.12) as in Exercise 4.
* Join the data and only keep counties that have bus stops.


In [183]:
#choosing a different set of ITP IDS
ITP_ID = [182, 183, 278, 154, 235, 232]

In [184]:
stops = (
    tbl.views.gtfs_schedule_dim_stops()
    >> select(_.itp_id == _.calitp_itp_id, _.stop_id, 
             _.stop_lat, _.stop_lon, _.stop_name)
    >> arrange(_.itp_id, _.stop_id, 
               _.stop_lat, _.stop_lon)
    >> collect()
    >> filter(_.itp_id.isin(ITP_ID))
    >> distinct()
)

In [185]:
#creating point geometry from longtitude &  latitude 
stops = shared_utils.geography_utils.create_point_geometry(stops, 'stop_lon', 'stop_lat')

In [186]:
stops.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry
0,154,19113,33.54,-117.78,Laguna Beach Bus Depot,POINT (-117.78271 33.54487)
1,154,19114,33.54,-117.78,Legion Hall,POINT (-117.78006 33.54069)


In [187]:
#checking CRS
stops.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [188]:
#bringing in CA dataframe
geojson = gpd.read_file('https://opendata.arcgis.com/datasets/8713ced9b78a4abb97dc130a691a8695_0.geojson').to_crs(epsg=4326)

In [189]:
#Joining the 2 dataframes, only keeping stops that are in a county
stops_joined = gpd.sjoin(stops, geojson, how='inner')

### Bring in a new table from BigQuery

* In `transitstacks`, there's a table called `ntd_stats`. 
* Decide what columns to keep.
* Merge `ntd_stats` with the `stops` table to create 1 df.

In [193]:
ntd_stats = (tbl.transitstacks.ntd_stats()
             >> select(_.transit_provider, _.itp_id, _.upt_total_2019)
             >> collect()
             >> filter(_.itp_id.isin(ITP_ID))
            )

In [226]:
#getting rid of commas & assigning to int to sum up later. 
ntd_stats = ntd_stats.assign(
    upt_total_2019 = ntd_stats.upt_total_2019.replace(',','', regex=True)).astype({"upt_total_2019": int}) 

### Merging the stops with ntd_stats - HELP
* Make sure all stop ids are unique
* When I dropped duplicates for stop ID to make sure I get only unique IDS, it dropped unique ones too? See above...

In [195]:
df2 = stops_joined.merge(ntd_stats, how = 'left', on ='itp_id')

In [196]:
f'out of the {len(df2)} rows, only {df2.stop_id.nunique()} are unique'

'out of the 44589 rows, only 22356 are unique'

In [197]:
#new dataframe with only unique stop ids & trips per stop
df3 = df2.drop_duplicates(subset=['stop_id'])

In [198]:
#checking that the rows make sense.
len(df3)

22356

In [225]:
#filtered for unique stops ids 
df3[(df3.COUNTY_NAME.str.contains("Riverside", case= False))]

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,transit_provider,upt_total_2019
31059,232,8815,33.99,-117.55,Goodman @ Goodman Nb Fs 01,POINT (-117.55461 33.99433),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,OmniTrans,10863530
31060,235,8341,33.90,-117.47,LA SIERRA METROLINK STATION,POINT (-117.46784 33.90066),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,Orange County Transportation Authority,40743654


In [191]:
#not filtered, five stop ids that are all unique?
stops_joined[(stops_joined.COUNTY_NAME.str.contains("Riverside", case= False))]

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area
19559,232,7593,33.98,-117.37,Riverside Metro Link Sb Lat,POINT (-117.37136 33.97539),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84
19746,232,8087,34.00,-117.05,County Line @ Fourth Wb Fs,POINT (-117.05295 34.00412),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84
19931,232,8603,33.98,-117.37,University @ Lemon Eb Ns,POINT (-117.37242 33.98087),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84
20096,232,8815,33.99,-117.55,Goodman @ Goodman Nb Fs 01,POINT (-117.55461 33.99433),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84
30531,235,8341,33.90,-117.47,LA SIERRA METROLINK STATION,POINT (-117.46784 33.90066),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84


### Aggregate 
<b> Instructions </b> 
* Write a function to aggregate to the operator level or county level, add new columns for desired metrics.
* Merge in CA shapefile to get a gdf.
* Add another `geometry` column, called `centroid`, and grab the county's centroid.
* Refer to [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.set_geometry.html) to see how to pick which column to use as the `geometry` for the gdf, since technically, a gdf can handle multiple geometry columns.



In [199]:
df3.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,transit_provider,upt_total_2019
0,154,19113,33.54,-117.78,Laguna Beach Bus Depot,POINT (-117.78271 33.54487),29,30,Orange,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,Laguna Beach Municipal Transit,820829
1,154,19114,33.54,-117.78,Legion Hall,POINT (-117.78006 33.54069),29,30,Orange,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,Laguna Beach Municipal Transit,820829


In [200]:
#County Subsets
county_stops = df3[['geometry','stop_id','COUNTY_NAME']]
county_passengers = df3[['geometry','upt_total_2019','COUNTY_NAME']]

In [201]:
#transit providers
transit_stops = df3[['geometry','stop_id','transit_provider']]
transit_passengers = df3[['geometry','upt_total_2019','transit_provider']]

In [202]:
#function 2
def aggregate2 (df1, df2, group_by_col):
    df1 = df1.dissolve(by = group_by_col, aggfunc= 'nunique').rename(columns = {'stop_id':'total_stops'})
    df2 = df2.dissolve(by = group_by_col, aggfunc= 'sum').rename(columns = {'upt_total_2019':'total_trips'})
    df3 = df1.merge(df2, how = 'inner', on = [group_by_col, 'geometry'])
    #find sum up total trips 
    df3['total_trips_sum'] = df3['total_trips'].sum() 
    #multiply total stops by total trips within a transit operator / county  
    df3['frequency'] = df3['total_stops']*df3['total_trips']
    #divide
    df3['weighted_avg'] = df3['frequency']/df3['total_trips_sum'] 
    #drop old cols
    #df3 = df3.drop(columns=['frequency','total_stops_sum'])
    df3 = df3.reset_index()
    return df3

In [203]:
county_agg = aggregate2(county_stops, county_passengers, 'COUNTY_NAME')
county_agg

,COUNTY_NAME,geometry,total_stops,total_trips,total_trips_sum,frequency,weighted_avg
0,Los Angeles,"MULTIPOINT (-118.84339 34.03152, -118.82113 34...",13179,3900836858318,4427319900956,51409128955772922,11611.79
1,Orange,"MULTIPOINT (-118.10607 33.74889, -118.10577 33...",5581,238891438439,4427319900956,1333253117928059,301.14
2,Riverside,"MULTIPOINT (-117.55461 33.99433, -117.46784 33...",2,51607184,4427319900956,103214368,0.00
3,San Bernardino,"MULTIPOINT (-117.73260 34.00136, -117.72986 34...",337,3661009610,4427319900956,1233760238570,0.28
4,San Diego,"MULTIPOINT (-117.27792 32.83915, -117.27775 32...",3204,273485413980,4427319900956,876247266391920,197.92
5,Ventura,"MULTIPOINT (-118.88283 34.18006, -118.88280 34...",53,10393573425,4427319900956,550859391525,0.12


In [204]:
3864886414724/13179

293260976.91205704

In [205]:
51607184/2

25803592.0

In [206]:
transit_agg= aggregate2(transit_stops, transit_passengers, 'transit_provider')
transit_agg

,transit_provider,geometry,total_stops,total_trips,total_trips_sum,frequency,weighted_avg
0,Laguna Beach Municipal Transit,"MULTIPOINT (-117.80188 33.54913, -117.79987 33...",94,77157926,4427319900956,7252845044,0.00
1,Los Angeles Department of Transportation,"MULTIPOINT (-118.88283 34.18006, -118.88280 34...",3033,58514689341,4427319900956,177475052771253,40.09
2,Los Angeles Metro,"MULTIPOINT (-118.86092 34.17506, -118.85796 34...",10185,3867429062385,4427319900956,39389765000391225,8896.98
3,OmniTrans,"MULTIPOINT (-117.75124 34.05917, -117.73260 34...",339,3682736670,4427319900956,1248447731130,0.28
4,Orange County Transportation Authority,"MULTIPOINT (-118.28073 33.96019, -118.28053 33...",5501,224130840654,4427319900956,1232943754437654,278.49
5,San Diego Metropolitan Transit System,"MULTIPOINT (-117.27792 32.83915, -117.27775 32...",3204,273485413980,4427319900956,876247266391920,197.92


#### HELP, why is this not turning into polygons & centroids incorrect? 
* For transit_agg: "/tmp/ipykernel_1946/1881472231.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation." 

In [207]:
transit_agg2 = gpd.sjoin(transit_agg, geojson, how='inner')
transit_agg2['centroid'] = transit_agg2.geometry.centroid
transit_agg2

/tmp/ipykernel_2717/1420666529.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



,transit_provider,geometry,total_stops,total_trips,total_trips_sum,frequency,weighted_avg,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,centroid
0,Laguna Beach Municipal Transit,"MULTIPOINT (-117.80188 33.54913, -117.79987 33...",94,77157926,4427319900956,7252845044,0.00,29,30,Orange,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,POINT (-117.76699 33.53023)
2,Los Angeles Metro,"MULTIPOINT (-118.86092 34.17506, -118.85796 34...",10185,3867429062385,4427319900956,39389765000391225,8896.98,29,30,Orange,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,POINT (-118.30227 34.05841)
4,Orange County Transportation Authority,"MULTIPOINT (-118.28073 33.96019, -118.28053 33...",5501,224130840654,4427319900956,1232943754437654,278.49,29,30,Orange,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,POINT (-117.88146 33.73851)
1,Los Angeles Department of Transportation,"MULTIPOINT (-118.88283 34.18006, -118.88280 34...",3033,58514689341,4427319900956,177475052771253,40.09,18,19,Los Angeles,LOS,19,19,037,None,{3B1E1D69-2B1A-464D-BA43-611C4201B219},5.18,1.00,POINT (-118.31409 34.03954)
2,Los Angeles Metro,"MULTIPOINT (-118.86092 34.17506, -118.85796 34...",10185,3867429062385,4427319900956,39389765000391225,8896.98,18,19,Los Angeles,LOS,19,19,037,None,{3B1E1D69-2B1A-464D-BA43-611C4201B219},5.18,1.00,POINT (-118.30227 34.05841)
3,OmniTrans,"MULTIPOINT (-117.75124 34.05917, -117.73260 34...",339,3682736670,4427319900956,1248447731130,0.28,18,19,Los Angeles,LOS,19,19,037,None,{3B1E1D69-2B1A-464D-BA43-611C4201B219},5.18,1.00,POINT (-117.41648 34.09590)
4,Orange County Transportation Authority,"MULTIPOINT (-118.28073 33.96019, -118.28053 33...",5501,224130840654,4427319900956,1232943754437654,278.49,18,19,Los Angeles,LOS,19,19,037,None,{3B1E1D69-2B1A-464D-BA43-611C4201B219},5.18,1.00,POINT (-117.88146 33.73851)
1,Los Angeles Department of Transportation,"MULTIPOINT (-118.88283 34.18006, -118.88280 34...",3033,58514689341,4427319900956,177475052771253,40.09,55,56,Ventura,VEN,56,56,111,None,{DFDE8E6B-742D-48C4-A4A0-4952E37D938B},3.00,0.47,POINT (-118.31409 34.03954)
2,Los Angeles Metro,"MULTIPOINT (-118.86092 34.17506, -118.85796 34...",10185,3867429062385,4427319900956,39389765000391225,8896.98,55,56,Ventura,VEN,56,56,111,None,{DFDE8E6B-742D-48C4-A4A0-4952E37D938B},3.00,0.47,POINT (-118.30227 34.05841)
3,OmniTrans,"MULTIPOINT (-117.75124 34.05917, -117.73260 34...",339,3682736670,4427319900956,1248447731130,0.28,32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,POINT (-117.41648 34.09590)


In [208]:
county_agg2 = pd.merge(county_agg, geojson,  on=['COUNTY_NAME']).drop(columns = ['geometry_x']).rename(columns = {'COUNTY_NAME':'County'}) 
county_agg2 = county_agg2.loc[county_agg2['ISLAND'] != 'Channel Islands'] 
county_agg2['centroid'] = county_agg2.geometry_y.centroid
county_agg2

/tmp/ipykernel_2717/223611155.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



,County,total_stops,total_trips,total_trips_sum,frequency,weighted_avg,OBJECTID,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,geometry_y,centroid
0,Los Angeles,13179,3900836858318,4427319900956,51409128955772922,11611.79,19,LOS,19,19,037,None,{3B1E1D69-2B1A-464D-BA43-611C4201B219},5.18,1.00,"MULTIPOLYGON (((-117.66733 34.79317, -117.6672...",POINT (-118.21689 34.36117)
3,Orange,5581,238891438439,4427319900956,1333253117928059,301.14,30,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,"MULTIPOLYGON (((-118.11526 33.74164, -118.1150...",POINT (-117.76121 33.70309)
4,Riverside,2,51607184,4427319900956,103214368,0.00,33,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,"MULTIPOLYGON (((-114.43559 34.07847, -114.4356...",POINT (-115.99382 33.74374)
5,San Bernardino,337,3661009610,4427319900956,1233760238570,0.28,36,SBD,36,36,071,None,{E4DF6870-0D0B-40C1-AD3B-A60A72792CFD},10.46,5.13,"MULTIPOLYGON (((-115.41359 35.62499, -115.3970...",POINT (-116.17852 34.84146)
6,San Diego,3204,273485413980,4427319900956,876247266391920,197.92,37,SDG,37,37,073,None,{414826EC-689E-4084-BD03-5195DF2748BF},4.73,1.06,"MULTIPOLYGON (((-117.24152 33.44879, -117.2413...",POINT (-116.73509 33.03422)
7,Ventura,53,10393573425,4427319900956,550859391525,0.12,56,VEN,56,56,111,None,{DFDE8E6B-742D-48C4-A4A0-4952E37D938B},3.00,0.47,"MULTIPOLYGON (((-119.35649 34.87366, -119.3564...",POINT (-119.07833 34.47124)


### Visualizations
* Make one chart for comparing trips per stop by operators, and another chart for comparing it by counties. Use a function to do this.
* Make 1 map for comparing trips per stop by counties. Use `shared_utils.map_utils` to do this.
* Visualizations should follow the Cal-ITP style guide.
* More on `folium` and `ipyleaflet`: https://github.com/jorisvandenbossche/geopandas-tutorial/blob/master/05-more-on-visualization.ipynb

#### Bar chart
* Make it into a function.
* Don't print out bar chart b/c this causes a file saving error

In [209]:
def scatter_chart(df, x_col, y_col):
    y_col_stripped = y_col.replace('_',' ')
    x_col_stripped = x_col.replace('_',' ')
    chart_title = f"{y_col_stripped} by {x_col_stripped}" 
    chart = (alt.Chart(df).mark_circle(size=500).encode(
                 x=alt.X(x_col, title=x_col_stripped),
                 y=alt.Y(y_col, title=y_col_stripped),   
                 color =alt.Color(x_col, scale=alt.Scale(range=altair_utils.CALITP_CATEGORY_BOLD_COLORS)),
                 tooltip = [alt.Tooltip(x_col),alt.Tooltip(y_col)]).interactive().properties(width=600,height=250,
                 title = chart_title)
            )
    #return chart
    display(chart)

In [210]:
transit_agg2.head(1)

,transit_provider,geometry,total_stops,total_trips,total_trips_sum,frequency,weighted_avg,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,centroid
0,Laguna Beach Municipal Transit,"MULTIPOINT (-117.80188 33.54913, -117.79987 33...",94,77157926,4427319900956,7252845044,0.00,29,30,Orange,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,POINT (-117.76699 33.53023)


In [211]:
operators_graph = transit_agg2[['transit_provider','weighted_avg']].drop_duplicates() 

In [212]:
scatter_chart(operators_graph, 'transit_provider','weighted_avg')

alt.Chart(...)

In [213]:
county_graph = county_agg2[['County','weighted_avg']].drop_duplicates() 

In [214]:
scatter_chart(county_graph, 'County','weighted_avg')

alt.Chart(...)

#### Map

In [215]:
choropleth_dict = ({
            "layer_name": 'my layer',
            "MIN_VALUE": int(0),
            "MAX_VALUE": int(100),
            "plot_col_name": 'County_Name',
            "fig_width": '100%',
            "fig_height": '100%',
            "fig_min_width_px": '600px',
            "fig_min_height_px": '600px'})

In [216]:
test = county_agg2.drop(columns = ['centroid']).rename(columns = {'geometry_y':'geometry'})
test

,County,total_stops,total_trips,total_trips_sum,frequency,weighted_avg,OBJECTID,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,geometry
0,Los Angeles,13179,3900836858318,4427319900956,51409128955772922,11611.79,19,LOS,19,19,037,None,{3B1E1D69-2B1A-464D-BA43-611C4201B219},5.18,1.00,"MULTIPOLYGON (((-117.66733 34.79317, -117.6672..."
3,Orange,5581,238891438439,4427319900956,1333253117928059,301.14,30,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,"MULTIPOLYGON (((-118.11526 33.74164, -118.1150..."
4,Riverside,2,51607184,4427319900956,103214368,0.00,33,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,"MULTIPOLYGON (((-114.43559 34.07847, -114.4356..."
5,San Bernardino,337,3661009610,4427319900956,1233760238570,0.28,36,SBD,36,36,071,None,{E4DF6870-0D0B-40C1-AD3B-A60A72792CFD},10.46,5.13,"MULTIPOLYGON (((-115.41359 35.62499, -115.3970..."
6,San Diego,3204,273485413980,4427319900956,876247266391920,197.92,37,SDG,37,37,073,None,{414826EC-689E-4084-BD03-5195DF2748BF},4.73,1.06,"MULTIPOLYGON (((-117.24152 33.44879, -117.2413..."
7,Ventura,53,10393573425,4427319900956,550859391525,0.12,56,VEN,56,56,111,None,{DFDE8E6B-742D-48C4-A4A0-4952E37D938B},3.00,0.47,"MULTIPOLYGON (((-119.35649 34.87366, -119.3564..."


In [217]:
colorscale = branca.colormap.StepColormap(
                colors=["gray", "green", "navy"], 
                #index=[2_000, 4_000, 8_000],
                #vmin=0, vmax=15_000,
)


In [218]:
def grab_region_centroids():
    # This parquet is created in shared_utils/shared_data.py
    df = pd.read_parquet(
        "gs://calitp-analytics-data/data-analyses/ca_county_centroids.parquet")
    
    df = df.assign(
        centroid = df.centroid.apply(lambda x: x.tolist())
    )    
    
    # Manipulate parquet file to be dictionary to use in map_utils
    region_centroids = dict(
        zip(df.county_name, 
            df[["centroid", "zoom"]].to_dict(orient="records")
           )
    )

    return region_centroids

In [219]:
REGION_CENTROIDS = grab_region_centroids()

In [220]:
shared_utils.map_utils.make_ipyleaflet_choropleth_map(test, 
                                                     plot_col = 'weighted_avg',
                                                     geometry_col = 'COUNTY_NAME', 
                                                     choropleth_dict = choropleth_dict, 
                                                     colorscale = colorscale,
                                                     zoom=REGION_CENTROIDS["CA"]["zoom"], centroid = REGION_CENTROIDS["CA"]["centroid"])

KeyError: 'Only a column name can be used for the key in a dtype mappings argument.'